# LeetCode 2202 — Maximize the Topmost Element After K Moves bằng Python

Link đề: https://leetcode.com/problems/maximize-the-topmost-element-after-k-moves/ · Độ khó: **Medium** · Chủ đề: Array, Greedy

Skeleton LeetCode:

```python
class Solution:
    def maximumTop(self, nums: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa đúng trạng thái của trò chơi "chồng đĩa": phần còn lại của chồng (một **hậu tố** của `nums` cộng các phần tử đã đặt lại) và **tập phần tử đã lấy ra**.
2. Viết **vét cạn có ghi nhớ** trên trạng thái để có đáp án chuẩn cho dữ liệu nhỏ, rồi dùng nó để **phát hiện quy luật**.
3. Rút ra **phân tích trường hợp** `O(min(n, k))`: phần tử đỉnh sau đúng `k` bước chỉ có thể là `max(nums[0..k−2])` (lấy ra `k−1` rồi đặt lại cái lớn nhất) hoặc `nums[k]` (lấy ra đúng `k` cái).
4. Xử lý các biên đặc biệt: `k = 0`, `n = 1` (tính **chẵn lẻ** của `k`), `k > n`.
5. Kiểm thử đối chiếu phân tích trường hợp với vét cạn trên **mọi** mảng nhỏ và mọi `k` nhỏ.

## 2. Đề bài & ví dụ

Mảng `nums` mô tả một chồng, `nums[0]` ở **trên cùng**. Mỗi **bước** bạn làm một trong hai việc:
- nếu chồng khác rỗng: **lấy ra** phần tử trên cùng;
- nếu đã có phần tử bị lấy ra: **đặt lại** một phần tử bất kỳ trong số đó lên đỉnh chồng.

Phải làm **đúng** `k` bước. Trả về giá trị **lớn nhất** có thể của phần tử trên cùng sau `k` bước; nếu không thể có chồng khác rỗng sau `k` bước, trả về `-1`.

**Ví dụ 1:** `nums = [5,2,2,4,0,6], k = 4` → `5`: lấy ra 5, 2, 2 (3 bước), đặt lại 5 (bước 4).

**Ví dụ 2:** `nums = [2], k = 1` → `-1`: bước duy nhất buộc phải lấy ra 2, chồng rỗng.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `0 ≤ nums[i], k ≤ 10⁹`.

In [1]:
import random
import time
from functools import lru_cache
from itertools import product
from typing import List

examples = [([5, 2, 2, 4, 0, 6], 4, 5), ([2], 1, -1)]
for nums, k, exp in examples:
    print(f"nums={nums}, k={k} -> kỳ vọng {exp}")

nums=[5, 2, 2, 4, 0, 6], k=4 -> kỳ vọng 5
nums=[2], k=1 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1 Ngăn xếp (stack) bằng list

Đỉnh chồng ứng với `nums[0]`. Để mô phỏng thuận tiện bằng `list`, ta đảo ngược: `stack = nums[::-1]`, đỉnh là `stack[-1]`; lấy ra `stack.pop()`, đặt lại `stack.append(x)`.

In [2]:
nums = [5, 2, 2, 4, 0, 6]
stack = nums[::-1]; removed = []
for _ in range(3):
    removed.append(stack.pop())
stack.append(max(removed))
print("đỉnh:", stack[-1], "| chồng (từ trên xuống):", stack[::-1], "| đã lấy ra còn:", removed)

đỉnh: 5 | chồng (từ trên xuống): [5, 4, 0, 6] | đã lấy ra còn: [5, 2, 2]


### 3.2 Trạng thái trò chơi và ghi nhớ

Sau một chuỗi bước, trạng thái gồm: **chồng hiện tại** (tuple) và **đa tập các phần tử đang ở ngoài** (tuple đã sắp xếp). Cùng một trạng thái với cùng số bước còn lại cho cùng tập kết quả ⇒ ghi nhớ bằng `lru_cache`.

In [3]:
@lru_cache(maxsize=None)
def demo(state, k):
    return k if k <= 0 else demo(state, k - 1)
print(demo(("a",), 5), demo.cache_info().currsize, "trạng thái được nhớ")

0 6 trạng thái được nhớ


### 3.3 `max` trên đoạn đầu và mảng rỗng

`max(nums[:k−1])` lỗi `ValueError` nếu đoạn rỗng (`k ≤ 1`). Dùng `max(seq, default=-1)` để có giá trị mặc định.

In [4]:
print(max([3, 1], default=-1), max([], default=-1))

3 -1


### 3.4 Tính chẵn lẻ khi chỉ có một phần tử

Với `n = 1`, chồng **luân phiên** rỗng / có phần tử: bước lẻ luôn là "lấy ra" (chồng rỗng), bước chẵn luôn là "đặt lại". Sau `k` bước: rỗng ⇔ `k` lẻ.

In [5]:
for k in range(6):
    print(f"n=1, k={k}: chồng {'rỗng -> -1' if k % 2 else 'có phần tử'}")

n=1, k=0: chồng có phần tử
n=1, k=1: chồng rỗng -> -1
n=1, k=2: chồng có phần tử
n=1, k=3: chồng rỗng -> -1
n=1, k=4: chồng có phần tử
n=1, k=5: chồng rỗng -> -1


## 4. Ý tưởng nền tảng / trực giác

Hãy hỏi: **phần tử nào có thể nằm trên đỉnh sau đúng `k` bước?** Xét bước **cuối cùng**:

- **Nếu bước cuối là "đặt lại"** phần tử `x`: `x` phải đã được lấy ra trước đó. Trong `k − 1` bước đầu ta chỉ có thể lấy ra tối đa `k − 1` phần tử **đầu tiên** của `nums` (và lấy theo thứ tự từ trên xuống). Vậy `x ∈ nums[0..k−2]`, và mọi phần tử như vậy đều đạt được: lấy ra `k − 1` phần tử đầu rồi đặt lại `x`. ⇒ ứng viên `max(nums[0..k−2])`.
- **Nếu bước cuối là "lấy ra"**: đỉnh mới là phần tử ngay dưới. Cách đơn giản: lấy ra đúng `k` phần tử đầu ⇒ đỉnh là `nums[k]` (nếu `k < n`). Có cách nào khác để kết thúc bằng "lấy ra" mà đỉnh là phần tử lớn hơn? Phần tử dưới đỉnh khi đó hoặc là `nums[j]` gốc, hoặc một phần tử đã đặt lại — nhưng để có phần tử đặt lại nằm **dưới** đỉnh ta tốn thêm bước; phân tích kỹ (và vét cạn ở mục 5) cho thấy không vượt quá hai ứng viên trên khi `n ≥ 2`.

Tính tay ví dụ 1 (`k = 4`): `max(nums[0..2]) = max(5, 2, 2) = 5`; `nums[4] = 0` ⇒ đáp án `5` ✓.

Các trường hợp đặc biệt: `k = 0` ⇒ `nums[0]`; `n = 1` ⇒ phụ thuộc chẵn lẻ; `k > n` ⇒ ta có thể lấy hết rồi đặt lại/lấy ra luân phiên để "đốt" bước thừa, đáp án `max(nums)` (khi `n ≥ 2`).

In [6]:
nums, k = [5, 2, 2, 4, 0, 6], 4
cand1 = max(nums[:k - 1], default=-1)
cand2 = nums[k] if k < len(nums) else -1
print("ứng viên đặt lại max(nums[:k-1]) =", cand1, "| ứng viên lấy ra đúng k: nums[k] =", cand2, "-> đáp án", max(cand1, cand2))

ứng viên đặt lại max(nums[:k-1]) = 5 | ứng viên lấy ra đúng k: nums[k] = 0 -> đáp án 5


## 5. Cách 1 — Brute Force / baseline: vét cạn mọi chuỗi bước có ghi nhớ

**Ý tưởng:** trạng thái `(stack, out)` với `stack` là tuple (đỉnh ở cuối) và `out` là tuple đã sắp các phần tử đang ở ngoài. Hàm `best(stack, out, k)` trả về đỉnh lớn nhất có thể (hoặc `-1`). Mỗi bước thử: lấy ra (nếu chồng khác rỗng) hoặc đặt lại **mỗi giá trị khác nhau** trong `out`.

In [7]:
def top_brute(nums, k):
    @lru_cache(maxsize=None)
    def best(stack, out, k):
        if k == 0:
            return stack[-1] if stack else -1
        res = -1
        if stack:                                            # lấy ra
            res = max(res, best(stack[:-1], tuple(sorted(out + (stack[-1],))), k - 1))
        for v in set(out):                                   # đặt lại v
            lst = list(out); lst.remove(v)
            res = max(res, best(stack + (v,), tuple(lst), k - 1))
        return res
    return best(tuple(nums[::-1]), (), k)

for nums, k, exp in examples:
    got = top_brute(nums, k); print(nums, k, "->", got); assert got == exp

[5, 2, 2, 4, 0, 6] 4 -> 5
[2] 1 -> -1


**Độ phức tạp:** số trạng thái tăng theo hàm mũ của `k` (mỗi bước có tới `1 + |out|` lựa chọn). Chỉ dùng cho `n ≤ 5`, `k ≤ 7`. Với `k` tới `10⁹` thì hoàn toàn không thể.

**Nút thắt:** ta mô phỏng chuỗi bước trong khi chỉ cần biết **tập các giá trị có thể nằm trên đỉnh** — một tập rất có cấu trúc. Hãy dùng vét cạn để quan sát tập đó.

In [8]:
def reachable_tops(nums, k):
    @lru_cache(maxsize=None)
    def go(stack, out, k):
        if k == 0:
            return frozenset([stack[-1]]) if stack else frozenset()
        res = set()
        if stack: res |= go(stack[:-1], tuple(sorted(out + (stack[-1],))), k - 1)
        for v in set(out):
            lst = list(out); lst.remove(v); res |= go(stack + (v,), tuple(lst), k - 1)
        return frozenset(res)
    return sorted(go(tuple(nums[::-1]), (), k))

nums = [10, 20, 30, 40, 50]
for k in range(0, 8):
    print(f"k={k}: các giá trị có thể ở đỉnh = {reachable_tops(nums, k)}")

k=0: các giá trị có thể ở đỉnh = [10]
k=1: các giá trị có thể ở đỉnh = [20]
k=2: các giá trị có thể ở đỉnh = [10, 30]
k=3: các giá trị có thể ở đỉnh = [10, 20, 40]
k=4: các giá trị có thể ở đỉnh = [10, 20, 30, 50]
k=5: các giá trị có thể ở đỉnh = [10, 20, 30, 40]
k=6: các giá trị có thể ở đỉnh = [10, 20, 30, 40, 50]
k=7: các giá trị có thể ở đỉnh = [10, 20, 30, 40, 50]


Quan sát bảng trên với `nums = [10, 20, 30, 40, 50]` (giá trị = vị trí × 10 cho dễ đọc):
- `k = 1`: chỉ `20` (= `nums[1]`).
- `k = 2`: `10` (đặt lại) và `30` (= `nums[2]`) — **không** có `20` = `nums[k−1]`.
- `k = 3`: `{10, 20}` (= `nums[0..1]`) và `40` (= `nums[3]`).
- `k ≥ n + 1`: mọi giá trị.

Quy luật: tập đỉnh = `nums[0..k−2] ∪ {nums[k]}` (khi `k < n`), riêng `nums[k−1]` **không bao giờ** đạt được khi `k ≥ 1`.

## 6. Ý tưởng thuật toán / bước đột phá

**Định lý (với `n ≥ 2`, `1 ≤ k`):** tập giá trị có thể nằm trên đỉnh sau đúng `k` bước là
- `{nums[i] : 0 ≤ i ≤ k − 2} ∪ {nums[k]}` nếu `k < n`;
- `{nums[i] : 0 ≤ i ≤ n − 2}` nếu `k = n` (lấy ra hết thì rỗng; nên chỉ còn đặt lại);
- **mọi** `nums[i]` nếu `k > n`.

**Quy tắc "nếu… thì…":**
- **Nếu** `k == 0` **thì** `nums[0]`.
- **Nếu** `n == 1` **thì** `-1` khi `k` lẻ, `nums[0]` khi `k` chẵn.
- **Nếu** `k > n` **thì** `max(nums)`.
- **Ngược lại** `max(max(nums[:k−1], default=−1), nums[k] if k < n else −1)`.

**Vì sao `nums[k−1]` bị loại:** để `nums[k−1]` nằm trên đỉnh ta phải lấy ra đúng `k − 1` phần tử trên nó (tốn `k − 1` bước), còn đúng 1 bước — bước đó buộc phải lấy ra nó (đỉnh đổi) hoặc đặt lại một phần tử khác lên trên nó. Không còn cách "đứng yên".

**Vì sao `k > n` cho mọi giá trị (khi `n ≥ 2`):** có thể "đốt" bước thừa bằng cặp (lấy ra, đặt lại) và chỉnh chẵn lẻ nhờ có ít nhất hai phần tử — vét cạn xác nhận ở mục 12.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Dựng tập đỉnh khả dĩ theo định lý, lấy max

Viết trực tiếp tập ứng viên (hữu ích để kiểm định lý với vét cạn ở mức **tập**, không chỉ max).

In [9]:
def candidate_tops(nums, k):
    n = len(nums)
    if k == 0:
        return {nums[0]}
    if n == 1:
        return set() if k % 2 else {nums[0]}
    if k > n:
        return set(nums)
    cand = set(nums[:k - 1])
    if k < n:
        cand.add(nums[k])
    return cand

def top_candidates(nums, k):
    c = candidate_tops(nums, k)
    return max(c) if c else -1

for nums, k, exp in examples:
    assert top_candidates(nums, k) == exp
print([top_candidates(nums, k) for nums, k, _ in examples])

[5, -1]


**Độ phức tạp Cách 2:** `O(min(n, k))` thời gian và bộ nhớ (tập ứng viên).

### 7.2 Cách 3 — Phân tích trường hợp, chỉ giữ max (bản nộp chính)

In [10]:
def top_cases(nums, k):
    n = len(nums)
    if k == 0:
        return nums[0]
    if n == 1:
        return -1 if k % 2 else nums[0]
    if k > n:
        return max(nums)
    best = max(nums[:k - 1], default=-1)       # lấy ra k-1 phần tử rồi đặt lại cái lớn nhất
    if k < n:
        best = max(best, nums[k])              # lấy ra đúng k phần tử
    return best

for nums, k, exp in examples:
    assert top_cases(nums, k) == exp
print([top_cases(nums, k) for nums, k, _ in examples])

[5, -1]


**Độ phức tạp Cách 3:** `O(min(n, k))` (slice + max), Space `O(min(n, k))` do slice; có thể `O(1)` bằng vòng lặp.

### 7.3 Cách 4 — Một vòng lặp `O(1)` bộ nhớ, gộp mọi trường hợp

Duyệt `i` từ `0` tới `min(n − 1, k)`: phần tử `nums[i]` là ứng viên nếu `i ≤ k − 2` hoặc `i == k`. Trường hợp `k > n` tự động gộp (mọi `i ≤ n − 1 ≤ k − 2`). Biên `k = 0` và `n = 1` xử lý riêng.

In [11]:
def top_loop(nums, k):
    n = len(nums)
    if n == 1:
        return -1 if k % 2 else nums[0]
    best = -1
    for i in range(min(n, k + 1)):
        if i <= k - 2 or i == k:
            if nums[i] > best:
                best = nums[i]
    return best

for nums, k, exp in examples:
    assert top_loop(nums, k) == exp
print([top_loop(nums, k) for nums, k, _ in examples], "| k=0:", top_loop([7, 9], 0))

[5, -1] | k=0: 7


**Độ phức tạp Cách 4:** `O(min(n, k))` thời gian, `O(1)` bộ nhớ. Lưu ý `k` tới `10⁹` nhưng vòng lặp chỉ chạy `min(n, k+1) ≤ 10⁵` lần.

## 8. TRACE từng bước

Trace Cách 4 trên ví dụ 1 `nums = [5,2,2,4,0,6], k = 4`: duyệt `i = 0..4`.

| i | nums[i] | `i ≤ k−2 = 2`? | `i == k`? | ứng viên? | best |
|---|---|---|---|---|---|
| 0 | 5 | ✓ | | ✓ | 5 |
| 1 | 2 | ✓ | | ✓ | 5 |
| 2 | 2 | ✓ | | ✓ | 5 |
| 3 | 4 | ✗ | ✗ | ✗ (đây là `nums[k−1]`) | 5 |
| 4 | 0 | ✗ | ✓ | ✓ | 5 |

Đáp án `5`. Đáng chú ý: `nums[3] = 4` bị loại dù lớn hơn `nums[4]`.

In [12]:
def trace_loop(nums, k):
    n = len(nums)
    if n == 1:
        print("n = 1 -> phụ thuộc chẵn lẻ của k"); return -1 if k % 2 else nums[0]
    best = -1
    for i in range(min(n, k + 1)):
        ok = i <= k - 2 or i == k
        if ok: best = max(best, nums[i])
        why = "đặt lại (i <= k-2)" if i <= k - 2 else ("lấy ra đúng k (i == k)" if i == k else "LOẠI: nums[k-1]")
        print(f"i={i}: nums[i]={nums[i]} -> {why:24s} best={best}")
    return best

print("Đáp án:", trace_loop([5, 2, 2, 4, 0, 6], 4))
print()
print("Đáp án:", trace_loop([1, 10, 2], 2))

i=0: nums[i]=5 -> đặt lại (i <= k-2)       best=5
i=1: nums[i]=2 -> đặt lại (i <= k-2)       best=5
i=2: nums[i]=2 -> đặt lại (i <= k-2)       best=5
i=3: nums[i]=4 -> LOẠI: nums[k-1]          best=5
i=4: nums[i]=0 -> lấy ra đúng k (i == k)   best=5
Đáp án: 5

i=0: nums[i]=1 -> đặt lại (i <= k-2)       best=1
i=1: nums[i]=10 -> LOẠI: nums[k-1]          best=1
i=2: nums[i]=2 -> lấy ra đúng k (i == k)   best=2
Đáp án: 2


In [13]:
# Xem một chuỗi bước cụ thể đạt đáp án (tái dựng từ vét cạn) cho ví dụ nhỏ
def show_sequence(nums, k):
    @lru_cache(maxsize=None)
    def best(stack, out, k):
        if k == 0: return (stack[-1] if stack else -1), ()
        options = []
        if stack:
            v, seq = best(stack[:-1], tuple(sorted(out + (stack[-1],))), k - 1)
            options.append((v, (f"lấy ra {stack[-1]}",) + seq))
        for x in sorted(set(out)):
            lst = list(out); lst.remove(x)
            v, seq = best(stack + (x,), tuple(lst), k - 1)
            options.append((v, (f"đặt lại {x}",) + seq))
        return max(options, key=lambda o: o[0]) if options else (-1, ())
    v, seq = best(tuple(nums[::-1]), (), k)
    print(f"nums={nums}, k={k}: đỉnh tốt nhất {v} qua chuỗi: {' -> '.join(seq)}")

show_sequence([5, 2, 2, 4], 4)
show_sequence([3, 1], 5)

nums=[5, 2, 2, 4], k=4: đỉnh tốt nhất 5 qua chuỗi: lấy ra 5 -> lấy ra 2 -> lấy ra 2 -> đặt lại 5
nums=[3, 1], k=5: đỉnh tốt nhất 3 qua chuỗi: lấy ra 3 -> lấy ra 1 -> đặt lại 1 -> lấy ra 1 -> đặt lại 3


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def maximumTop(self, nums: List[int], k: int) -> int:
        n = len(nums)
        if k == 0:
            return nums[0]
        if n == 1:                         # chồng luân phiên rỗng / có phần tử
            return -1 if k % 2 else nums[0]
        if k > n:                          # thừa bước: mọi phần tử đều đưa lên đỉnh được
            return max(nums)
        best = max(nums[:k - 1], default=-1)   # lấy ra k-1 phần tử, đặt lại cái lớn nhất
        if k < n:
            best = max(best, nums[k])          # lấy ra đúng k phần tử
        return best

for nums, k, exp in examples:
    assert Solution().maximumTop(nums, k) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Giả sử `n ≥ 2` và `k ≥ 1` (hai trường hợp `k = 0`, `n = 1` đã hiển nhiên). Ký hiệu `T(k)` là tập giá trị có thể nằm trên đỉnh sau đúng `k` bước.

**Nhận xét nền.** Chồng chỉ lấy từ trên xuống, nên tại mọi thời điểm các phần tử **gốc** đã từng bị lấy ra luôn là một **tiền tố** `nums[0..t−1]` (với `t` = số lần lấy ra phần tử gốc). Mỗi lần "lấy ra" một phần tử gốc tốn đúng 1 bước, nên `t ≤ k`.

**(a) Các ứng viên đều đạt được (khi `k ≤ n`).**
- Với `i ≤ k − 2`: lấy ra `k − 1` phần tử đầu (`k − 1 ≤ n − 1`, hợp lệ), bước thứ `k` đặt lại `nums[i]`.
- Với `k < n`: lấy ra `k` phần tử đầu, đỉnh là `nums[k]`.

**(b) Không còn giá trị nào khác (khi `k ≤ n`).** Xét trạng thái cuối, đỉnh là `x`.
- *Trường hợp `x` là một phần tử đã từng bị lấy ra rồi đặt lại:* nó thuộc tiền tố `nums[0..t−1]`, và ít nhất một bước là "đặt lại" ⇒ `t ≤ k − 1` ⇒ `x ∈ nums[0..k−2]`.
- *Trường hợp `x` là phần tử gốc `nums[t]` chưa từng bị lấy ra:* mọi phần tử đặt lại sau đó đều phải bị lấy ra lại (nếu không chúng nằm trên `nums[t]`). Mỗi "đặt lại" đi kèm đúng một "lấy ra lại" ⇒ `k = t + 2p`. Nếu `p = 0` thì `t = k` ⇒ `x = nums[k]`. Nếu `p ≥ 1` thì `t ≤ k − 2` ⇒ `x = nums[t]` với `t ≤ k − 2` — giá trị này đã nằm trong tập "đặt lại" ở (a).
- Vì vậy `nums[k−1]` (vị trí `t = k − 1`) không thể là đỉnh: nó đòi `k − (k−1) = 1 = 2p` (vô lý) hoặc đòi đã bị lấy ra (cần `t ≥ k`) rồi thêm một bước đặt lại (tổng ≥ `k + 1` bước).

**(c) `k > n`.** Lấy ra hết `n` phần tử còn thừa `r = k − n ≥ 1` bước, chồng đang rỗng và cả `n ≥ 2` phần tử ở ngoài. Muốn kết thúc với `x` trên đỉnh: **nếu `r` lẻ** — đặt lại `x` (1 bước), rồi đốt `r − 1` bước (số chẵn) bằng các cặp (lấy ra `x`, đặt lại `x`); **nếu `r` chẵn** — đặt lại một phần tử `y ≠ x` (tồn tại vì `n ≥ 2`) rồi đặt lại `x` (2 bước), phần còn lại `r − 2` (chẵn) đốt bằng các cặp (lấy ra `x`, đặt lại `x`). Vậy mọi giá trị đều đạt được: `T(k) = set(nums)`.

**(d) Kiểm chứng máy.** Ô dưới so `T(k)` của vét cạn với định lý trên mọi mảng giá trị phân biệt độ dài 2..5 và mọi `k ≤ 9`.

In [15]:
checked = 0
for n in range(2, 6):
    base = [10 * (i + 1) for i in range(n)]
    for k in range(1, 10):
        assert set(reachable_tops(base, k)) == candidate_tops(base, k)
        checked += 1
print(f"Định lý đúng trên {checked} bộ (n = 2..5, k = 1..9)")
show_sequence([3, 1], 4)          # k > n, r = 2 chẵn
show_sequence([3, 1, 2], 6)       # k > n, r = 3 lẻ

Định lý đúng trên 36 bộ (n = 2..5, k = 1..9)
nums=[3, 1], k=4: đỉnh tốt nhất 3 qua chuỗi: lấy ra 3 -> lấy ra 1 -> đặt lại 1 -> đặt lại 3
nums=[3, 1, 2], k=6: đỉnh tốt nhất 3 qua chuỗi: lấy ra 3 -> lấy ra 1 -> lấy ra 2 -> đặt lại 1 -> lấy ra 1 -> đặt lại 3


## 11. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn có ghi nhớ | hàm mũ theo `k` | hàm mũ |
| 2. Tập ứng viên | `O(min(n, k))` | `O(min(n, k))` |
| 3. Phân tích trường hợp | `O(min(n, k))` | `O(min(n, k))` (slice) |
| 4. Một vòng lặp | `O(min(n, k))` | `O(1)` |

Với `k = 10⁹`, các cách 2–4 vẫn chỉ tốn `O(n)`.

In [16]:
rng = random.Random(2202)
big = [rng.randint(0, 10**9) for _ in range(10**5)]
for k in (0, 1, 50000, 99999, 10**5, 10**5 + 1, 10**9):
    t = time.perf_counter()
    r3, r4 = top_cases(big, k), top_loop(big, k)
    assert r3 == r4 == Solution().maximumTop(big, k)
    print(f"k={k:>10}: {r3} ({time.perf_counter() - t:.3f}s)")

k=         0: 17961412 (0.000s)
k=         1: 841632643 (0.000s)
k=     50000: 999969146 (0.003s)
k=     99999: 999996204 (0.006s)
k=    100000: 999996204 (0.006s)
k=    100001: 999996204 (0.005s)
k=1000000000: 999996204 (0.005s)


## 12. Bộ test

- Ví dụ đề.
- Biên: `k = 0`; `n = 1` với `k` chẵn/lẻ; `k = n − 1`, `k = n`, `k = n + 1`; `nums[k−1]` là phần tử lớn nhất (bẫy).
- **Kiểm ở mức tập**: với mọi mảng độ dài `1..4` trên `{0, 1, 2, 3}` (giá trị khác nhau theo vị trí để phân biệt) và mọi `k ≤ 7`, tập đỉnh của vét cạn == tập theo định lý; và mọi cách cho cùng max.

In [17]:
fns = {"brute": top_brute, "candidates": top_candidates, "cases": top_cases, "loop": top_loop,
       "Solution": lambda a, k: Solution().maximumTop(a, k)}
edge = [([7], 0, 7), ([7], 1, -1), ([7], 2, 7), ([7], 3, -1), ([1, 10, 2], 2, 2), ([1, 10], 2, 1),
        ([1, 10], 3, 10), ([4, 3, 9], 3, 4), ([4, 3, 9], 4, 9), ([0, 0], 1, 0)]
for nums, k, exp in [(a, k, e) for a, k, e in examples] + edge:
    for name, fn in fns.items():
        assert fn(nums, k) == exp, (name, nums, k, fn(nums, k))
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [18]:
cnt = 0
for n in range(1, 5):
    base = [10 * (i + 1) for i in range(n)]                 # giá trị phân biệt theo vị trí
    for k in range(0, 8):
        assert set(reachable_tops(base, k)) == candidate_tops(base, k), (base, k)
        cnt += 1
rng = random.Random(42)
for _ in range(300):
    nums = [rng.randint(0, 5) for _ in range(rng.randint(1, 4))]; k = rng.randint(0, 7)
    want = top_brute(nums, k)
    for name, fn in fns.items():
        assert fn(nums, k) == want, (name, nums, k)
print(f"Định lý về tập đỉnh đúng trên {cnt} bộ (n <= 4, k <= 7); 300 test ngẫu nhiên khớp vét cạn")

Định lý về tập đỉnh đúng trên 32 bộ (n <= 4, k <= 7); 300 test ngẫu nhiên khớp vét cạn


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tính cả `nums[k−1]` (dùng `max(nums[:k])`)

In [19]:
def wrong_include_k_minus_1(nums, k):
    n = len(nums)
    if k == 0: return nums[0]
    if n == 1: return -1 if k % 2 else nums[0]
    if k > n: return max(nums)
    best = max(nums[:k])                                     # SAI: gồm cả nums[k-1]
    if k < n: best = max(best, nums[k])
    return best

a, k = [1, 10, 2], 2
print("sai:", wrong_include_k_minus_1(a, k), "| đúng:", top_brute(a, k))

sai: 10 | đúng: 2


In [20]:
print("sửa (nums[:k-1]):", Solution().maximumTop(a, k))

sửa (nums[:k-1]): 2


### Lỗi 2 — Quên trường hợp `n = 1` (chẵn lẻ)

Với `n = 1`, công thức chung cho `k > n` trả `max(nums)` dù `k` lẻ (chồng rỗng).

In [21]:
def wrong_no_single(nums, k):
    n = len(nums)
    if k == 0: return nums[0]
    if k > n: return max(nums)                               # SAI khi n == 1 và k lẻ
    best = max(nums[:k - 1], default=-1)
    if k < n: best = max(best, nums[k])
    return best

print("sai:", wrong_no_single([5], 3), "| đúng:", top_brute([5], 3))

sai: 5 | đúng: -1


In [22]:
print("sửa:", Solution().maximumTop([5], 3), "| n=1, k=4:", Solution().maximumTop([5], 4))

sửa: -1 | n=1, k=4: 5


### Lỗi 3 — Cho rằng `k ≥ n` thì luôn `-1`

Chỉ đúng khi `n = 1` và `k` lẻ. Với `n ≥ 2`, có thể đặt lại phần tử bất kỳ.

In [23]:
def wrong_k_ge_n(nums, k):
    n = len(nums)
    if k == 0: return nums[0]
    if k >= n: return -1                                     # SAI
    return max(max(nums[:k - 1], default=-1), nums[k])

a = [4, 3, 9]
print("sai:", [wrong_k_ge_n(a, k) for k in (3, 4, 5)], "| đúng:", [top_brute(a, k) for k in (3, 4, 5)])

sai: [-1, -1, -1] | đúng: [4, 9, 9]


In [24]:
print("sửa:", [Solution().maximumTop(a, k) for k in (3, 4, 5)])

sửa: [4, 9, 9]


### Lỗi 4 — Quên `k = 0` (hoặc lấy `max(nums[:-1])` khi `k = 0`)

`k = 0` nghĩa là không làm gì: đỉnh vẫn là `nums[0]`. Công thức chung với `k = 0` cho `nums[:-1]` (mọi phần tử trừ cuối!) do chỉ số âm.

In [25]:
def wrong_k_zero(nums, k):
    n = len(nums)
    if n == 1: return -1 if k % 2 else nums[0]
    if k > n: return max(nums)
    best = max(nums[:k - 1], default=-1)                     # k = 0 -> nums[:-1] !
    if k < n: best = max(best, nums[k])
    return best

a = [1, 9, 3]
print("sai:", wrong_k_zero(a, 0), "| đúng:", top_brute(a, 0))

sai: 9 | đúng: 1


In [26]:
print("sửa (xử lý k == 0 riêng):", Solution().maximumTop(a, 0))

sửa (xử lý k == 0 riêng): 1


### Lỗi 5 — Truy cập `nums[k]` khi `k == n`

Khi `k == n`, lấy ra đúng `k` phần tử làm chồng rỗng — không có `nums[k]`. Quên điều kiện `k < n` gây `IndexError` (ta bắt lỗi để xem).

In [27]:
def wrong_index(nums, k):
    try:
        return max(max(nums[:k - 1], default=-1), nums[k])  # SAI khi k == n
    except IndexError as e:
        return f"IndexError: {e}"

print("sai:", wrong_index([4, 3, 9], 3), "| đúng:", Solution().maximumTop([4, 3, 9], 3))

sai: IndexError: list index out of range | đúng: 4


In [28]:
print("sửa (chỉ thêm nums[k] khi k < n):", top_cases([4, 3, 9], 3))

sửa (chỉ thêm nums[k] khi k < n): 4


### Lỗi 6 — "Trong `k` bước nhìn thấy được `k + 1` phần tử đầu, vậy lấy max của chúng"

Trực giác "đi sâu được tới `nums[k]` thì mọi phần tử từ `nums[0]` tới `nums[k]` đều đưa lên đỉnh được" là sai ở đúng một chỗ: `nums[k−1]`. Bản sai dưới đây dùng `max(nums[:k+1])` cho mọi `k ≤ n − 1`. Ta để máy tìm phản ví dụ ngắn nhất.

In [29]:
def wrong_window_max(nums, k):
    n = len(nums)
    if k == 0: return nums[0]
    if n == 1: return -1 if k % 2 else nums[0]
    if k > n: return max(nums)
    return max(nums[:k + 1])                                  # SAI: gồm cả nums[k-1]

bad = next((list(t), k) for n in range(2, 5) for t in product(range(3), repeat=n) for k in range(1, n + 1)
           if wrong_window_max(list(t), k) != top_brute(list(t), k))
print("phản ví dụ ngắn nhất:", bad, "| sai:", wrong_window_max(*bad), "| đúng:", top_brute(*bad))

phản ví dụ ngắn nhất: ([0, 1], 2) | sai: 1 | đúng: 0


In [30]:
print("sửa:", Solution().maximumTop(*bad))

sửa: 0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Bài "đúng k bước" với thao tác có thể hoàn tác** ⇒ nghĩ về **bước cuối cùng** và **tính chẵn lẻ** của số bước thừa.
- `k` rất lớn (`10⁹`) nhưng chỉ `n` phần tử ⇒ đáp án thường là phân tích trường hợp `O(n)`.
- Dùng **vét cạn trạng thái** trên dữ liệu nhỏ để in ra tập kết quả và **đoán quy luật** trước khi chứng minh.
- Liệt kê biên: `k = 0`, `n = 1`, `k = n − 1, n, n + 1`.
- `max(..., default=...)` tránh lỗi trên dãy rỗng; cẩn thận slice với chỉ số âm (`nums[:-1]`).

## 15. Biến thể của bài

1. **Tối thiểu phần tử đỉnh** sau đúng `k` bước: cùng tập ứng viên, lấy `min`.
2. **Được làm tối đa `k` bước** (không bắt buộc đúng `k`): tập đỉnh = `nums[0..k]` ∪ ... (bài tập 5).
3. **Chỉ được đặt lại phần tử vừa lấy ra gần nhất** (như stack thứ hai): tập đỉnh thay đổi — phân tích lại bằng vét cạn.
4. **LeetCode 1033 / 2029 (game theory)**: các bài cần phân tích tính chẵn lẻ của số bước.

Minh họa biến thể 1 (tự dựng): min đỉnh, kiểm bằng vét cạn trên tập.

In [31]:
def min_top(nums, k):
    c = candidate_tops(nums, k)
    return min(c) if c else -1

rng = random.Random(1)
for _ in range(300):
    nums = [rng.randint(0, 9) for _ in range(rng.randint(1, 4))]; k = rng.randint(0, 7)
    tops = reachable_tops(nums, k)
    assert min_top(nums, k) == (min(tops) if tops else -1)
print("Biến thể min đỉnh khớp vét cạn; [5,2,2,4,0,6], k=4 ->", min_top([5, 2, 2, 4, 0, 6], 4))

Biến thể min đỉnh khớp vét cạn; [5,2,2,4,0,6], k=4 -> 0


Minh họa biến thể 3 (tự dựng) — **chỉ được đặt lại phần tử vừa lấy ra gần nhất** (các phần tử lấy ra xếp thành một chồng thứ hai, chỉ lấy được đỉnh của nó). Ta không chứng minh công thức mà dùng vét cạn để quan sát tập đỉnh: lúc này chuỗi bước giống như "di chuyển một con trỏ" lên xuống, nên đỉnh sau `k` bước là `nums[t]` với `t ≤ k` và `t ≡ k (mod 2)`.

In [32]:
def reachable_tops_lifo(nums, k):
    @lru_cache(maxsize=None)
    def go(t, k):                      # t = số phần tử đã lấy ra (chồng ngoài là nums[t-1], ..., nums[0])
        if k == 0:
            return frozenset([nums[t]]) if t < len(nums) else frozenset()
        res = set()
        if t < len(nums): res |= go(t + 1, k - 1)      # lấy ra
        if t > 0: res |= go(t - 1, k - 1)              # đặt lại phần tử vừa lấy ra gần nhất
        return frozenset(res)
    return sorted(go(0, k))

base = [10, 20, 30, 40, 50]
for k in range(0, 7):
    guess = sorted({base[t] for t in range(0, min(k, len(base) - 1) + 1) if (k - t) % 2 == 0})
    print(f"k={k}: đỉnh khả dĩ = {reachable_tops_lifo(base, k)} | dự đoán t ≡ k (mod 2): {guess}")
    assert reachable_tops_lifo(base, k) == guess

k=0: đỉnh khả dĩ = [10] | dự đoán t ≡ k (mod 2): [10]
k=1: đỉnh khả dĩ = [20] | dự đoán t ≡ k (mod 2): [20]
k=2: đỉnh khả dĩ = [10, 30] | dự đoán t ≡ k (mod 2): [10, 30]
k=3: đỉnh khả dĩ = [20, 40] | dự đoán t ≡ k (mod 2): [20, 40]
k=4: đỉnh khả dĩ = [10, 30, 50] | dự đoán t ≡ k (mod 2): [10, 30, 50]
k=5: đỉnh khả dĩ = [20, 40] | dự đoán t ≡ k (mod 2): [20, 40]
k=6: đỉnh khả dĩ = [10, 30, 50] | dự đoán t ≡ k (mod 2): [10, 30, 50]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn trạng thái | mô phỏng mọi chuỗi bước | hàm mũ | hàm mũ | khám phá quy luật, kiểm tra |
| 2. Tập ứng viên | dựng tập theo định lý | `O(min(n,k))` | `O(min(n,k))` | kiểm định lý ở mức tập |
| 3. Phân tích trường hợp | max hai ứng viên | `O(min(n,k))` | `O(min(n,k))` | **nộp bài**, dễ đọc |
| 4. Một vòng lặp | điều kiện `i ≤ k−2 or i == k` | `O(min(n,k))` | `O(1)` | tối ưu bộ nhớ |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `maximumTop([3, 8, 1, 9], 3)` và `maximumTop([3, 8, 1, 9], 1)`.
2. **Tính tay:** `maximumTop([2, 7, 4], k)` với `k = 0..5`.
3. **Điền TODO:** `top_set(nums, k)` trả về **tập** giá trị có thể ở đỉnh (theo định lý).
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `maximum_top_at_most(nums, k)` — được làm **tối đa** `k` bước (từ 0 tới `k`). Kiểm bằng vét cạn (`max` trên mọi `t ≤ k`).

In [33]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def top_set(nums, k):
    # TODO
    pass

In [34]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_maximum_top(nums, k):
    # TODO
    pass

In [35]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def maximum_top_at_most(nums, k):
    # TODO
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) `k = 3`: ứng viên `nums[0..1] = {3, 8}` và `nums[3] = 9`. (2) Đáp án `9`. (3) `k = 1`: chỉ `nums[1] = 8`.

**Bài 2.** (1) `k=0 → 2`, `k=1 → 7`. (2) `k=2`: `{2} ∪ {4}` → 4. `k=3 = n`: `{2, 7}` → 7. (3) `k ≥ 4 > n`: `max = 7`.

**Bài 3.** (1) Xử lý `k = 0`, `n = 1`, `k > n`. (2) `set(nums[:k−1])`. (3) Thêm `nums[k]` nếu `k < n`.

**Bài 4.** (1) Bốn nhánh `if`. (2) `max(..., default=−1)`. (3) Chỉ thêm `nums[k]` khi `k < n`.

**Bài 5.** (1) Hợp các tập đỉnh `T(t)` với `t = 0..k`. (2) `T(t − 1)` chứa `nums[t−1]` (lấy ra `t − 1` phần tử) nên khi được dừng sớm, `nums[k−1]` không còn bị loại. (3) Từ `t = n + 1` trở đi `T(t)` đã là toàn bộ `set(nums)` (khi `n ≥ 2`), nên chỉ cần hợp tới `t = min(k, n + 2)`.

## 19. Đáp án tham khảo

In [36]:
# Bài 1, 2
assert Solution().maximumTop([3, 8, 1, 9], 3) == 9 and Solution().maximumTop([3, 8, 1, 9], 1) == 8
r2 = [Solution().maximumTop([2, 7, 4], k) for k in range(6)]
assert r2 == [top_brute([2, 7, 4], k) for k in range(6)]
print("Bài 1: 9 và 8 | Bài 2:", r2)

Bài 1: 9 và 8 | Bài 2: [2, 7, 4, 7, 7, 7]


In [37]:
# Bài 3, 4
top_set_ans = candidate_tops
my_maximum_top_ans = top_cases
rng = random.Random(3)
for _ in range(300):
    base = [rng.randint(0, 50) for _ in range(rng.randint(1, 4))]; k = rng.randint(0, 7)
    assert set(reachable_tops(base, k)) == top_set_ans(base, k)
    assert my_maximum_top_ans(base, k) == top_brute(base, k)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [38]:
# Bài 5
def maximum_top_at_most_ans(nums, k):
    n = len(nums); s = set()
    for t in range(0, min(k, n + 2) + 1):          # các t lớn hơn không thêm giá trị mới (tập đã bão hòa)
        s |= candidate_tops(nums, t)
    return max(s) if s else -1

def at_most_brute(nums, k):
    best = -1
    for t in range(k + 1):
        best = max(best, top_brute(nums, t))
    return best

rng = random.Random(5)
for _ in range(200):
    nums = [rng.randint(0, 9) for _ in range(rng.randint(1, 4))]; k = rng.randint(0, 7)
    assert maximum_top_at_most_ans(nums, k) == at_most_brute(nums, k)
print("Bài 5: OK; [1,10,2], k=2 ->", maximum_top_at_most_ans([1, 10, 2], 2), "(làm 1 bước để 10 lên đỉnh)")

Bài 5: OK; [1,10,2], k=2 -> 10 (làm 1 bước để 10 lên đỉnh)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
if k == 0: return nums[0]
if n == 1: return -1 if k % 2 else nums[0]
if k > n: return max(nums)
best = max(nums[:k-1], default=-1)     # đặt lại cái lớn nhất trong k-1 phần tử đầu
if k < n: best = max(best, nums[k])    # lấy ra đúng k phần tử
return best
```

In [39]:
def template(nums, k):
    n = len(nums)
    if k == 0: return nums[0]
    if n == 1: return -1 if k % 2 else nums[0]
    if k > n: return max(nums)
    best = max(nums[:k - 1], default=-1)
    if k < n: best = max(best, nums[k])
    return best
print([template(a, k) for a, k, _ in examples])

[5, -1]


**Checklist:**

- [ ] `k == 0` ⇒ `nums[0]` (tránh slice `nums[:-1]`).
- [ ] `n == 1` ⇒ chẵn lẻ của `k`.
- [ ] `k > n` ⇒ `max(nums)`.
- [ ] Ứng viên "đặt lại" là `nums[0..k−2]` — **không** gồm `nums[k−1]`.
- [ ] Ứng viên "lấy ra đúng k" là `nums[k]` chỉ khi `k < n`.
- [ ] Đã kiểm bằng vét cạn trạng thái trên mảng nhỏ.